In [1]:
# Linear Regression exercises using Boston dataset from ISLP
# Goal: predict medv (median house value) from other factors including lstat (percent low SES households)
# File created: 25 November 2026
# Last edited: 25 November 2026

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.stats import norm

np.set_printoptions(suppress = True)
import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("Datasets/Boston.csv")
data.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     506 non-null    float64
 1   zn       506 non-null    float64
 2   indus    506 non-null    float64
 3   chas     506 non-null    int64  
 4   nox      506 non-null    float64
 5   rm       506 non-null    float64
 6   age      506 non-null    float64
 7   dis      506 non-null    float64
 8   rad      506 non-null    int64  
 9   tax      506 non-null    int64  
 10  ptratio  506 non-null    float64
 11  black    506 non-null    float64
 12  lstat    506 non-null    float64
 13  medv     506 non-null    float64
dtypes: float64(11), int64(3)
memory usage: 55.5 KB


In [5]:
data.columns

Index(['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax',
       'ptratio', 'black', 'lstat', 'medv'],
      dtype='object')

In [6]:
# Predicting "medv" from "lstat" using sklearn.linear_model's LinearRegression - see ISLP pages 117-118
print("Simple Linear Regression: medv = b0 + b1 x lstat + eps")
Y = data["medv"].to_numpy()
X = data["lstat"].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = LinearRegression()
model.fit(X, Y)
beta0 = model.intercept_
beta1 = model.coef_.item()
print("beta0 = %s, beta1 = %s" % (beta0, beta1))
Rsquare = model.score(X, Y)
print("Rsquare = %s" % Rsquare)

Simple Linear Regression: medv = b0 + b1 x lstat + eps
(506, 1) (506,)
beta0 = 34.55384087938312, beta1 = -0.9500493537579914
Rsquare = 0.5441462975864799


In [7]:
class Jie_SimpleLinearRegression:
    
    def __init__(self):
        self.beta0 = None
        self.beta1 = None
        self.sigma2 = None
        self.Rsquare = None
        self.beta0_stdev = None
        self.beta1_stdev = None
        self.beta0_t = None
        self.beta1_t = None
        self.beta0_p = None
        self.beta1_p = None
        self.results = None
    
    def fit(self, X, Y, predictor_name):
        
        Xc = np.insert(X, 0, np.ones(len(X)), axis = 1)
        self.beta0, self.beta1 = np.linalg.inv(Xc.T @ Xc) @ Xc.T @ Y
        
        Yhat = self.predict(X)
        self.sigma2 = (np.linalg.norm(Y - Yhat) ** 2) / (len(Y) - 2)
        
        beta_var = np.diag(self.sigma2 * np.linalg.inv(Xc.T @ Xc))
        self.beta0_stdev, self.beta1_stdev = np.sqrt(beta_var)

        self.beta0_t = self.beta0 / self.beta0_stdev
        self.beta1_t = self.beta1 / self.beta1_stdev
        
        self.beta0_p = 2 * norm.sf(np.abs(self.beta0_t))
        self.beta1_p = 2 * norm.sf(np.abs(self.beta1_t))

        beta = np.array([self.beta0, self.beta1])
        beta_stdev = np.array([self.beta0_stdev, self.beta1_stdev])
        beta_t = np.array([self.beta0_t, self.beta1_t])
        beta_p = np.array([self.beta0_p, self.beta1_p])
        self.results = pd.DataFrame(zip(beta, beta_stdev, beta_t, beta_p),
                                    columns = ["beta", "beta stdev", "beta t", "beta p"],
                                    index = ["intercept", predictor_name])
        print(self.results)
        
    def score(self, X, Y):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = self.predict(X)
        RSS = np.linalg.norm(Y - Yhat) ** 2
        TSS = np.linalg.norm(Y - np.mean(Y)) ** 2
        self.Rsquare = 1 - RSS/TSS
        print("Rsquare = %s" % self.Rsquare)
        
    def predict(self, X):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = (self.beta0 + self.beta1 * X).reshape(-1)
        return Yhat

In [8]:
# Predicting "medv" from "lstat" by hand - see ISLP pages 117-118
print("Simple Linear Regression: medv = b0 + b1 x lstat + eps")
Y = data["medv"].to_numpy()
predictor_name = "lstat"
X = data[predictor_name].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = Jie_SimpleLinearRegression()
model.fit(X, Y, predictor_name)
model.score(X, Y)

Simple Linear Regression: medv = b0 + b1 x lstat + eps
(506, 1) (506,)
                beta  beta stdev     beta t         beta p
intercept  34.553841    0.562627  61.415146   0.000000e+00
lstat      -0.950049    0.038733 -24.527900  7.445605e-133
Rsquare = 0.5441462975864797
